# Project 2: Technical Report
## Social Media and Mental Health
Haoran Chen | DTSC 2301 | Spring 2026

Portfolio Post: [project2.html](https://hchen34-creater.github.io/dtsc2301-portfolio/project2.html)

GitHub Repo: [github.com/hchen34-creater/dtsc2301-portfolio](https://github.com/hchen34-creater/dtsc2301-portfolio)

## Introduction

For this project I wanted to see if you could predict someones mental health based on how they use social media. I was curious about this because its something that effects alot of people my age and theres been alot of news about it lately. On March 25 2026 a jury in LA found Meta and Google liable in a lawsuit where a 20 year old woman said social media caused her anxiety and body dysmorphia. The jury ordered them to pay $6 million. Theres like 1500 more cases waiting after that one.

My partner helped me narrow down the topic since it was too broad at first. We decided to focus on stuff like stress, sleep, distraction, and comparing yourself to others since those are the things that actually affect school performance.

Research question: can we predict whether a person has high or low mental health impact based on their social media usage patterns, and which behaviors matter the most.

## Dataset

I used the "Social Media and Mental Health" dataset from Kaggle by Souvik Ahmed. Its survey data with 481 responses and 20 questions. Link: https://www.kaggle.com/datasets/souvikahmed071/social-media-and-mental-health

It covers demographics (age, gender), social media habits (time spent, platforms, purposeless scrolling), and mental health stuff rated on a 1-5 scale (how distracted they are, sleep issues, depression, comparing themselves to others, etc).

I picked this one because it had both social media usage data and mental health data in the same dataset so I could actually build a model to predict one from the other.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
df = pd.read_csv('smmh.csv')

# the column names are the full survey questions which makes the code impossible to read
# so im renaming them all
df.columns = [
    'Timestamp', 'Age', 'Gender', 'Relationship', 'Occupation', 'Organization',
    'Uses_SM', 'Platforms', 'Avg_Time', 'Purposeless', 'Distracted_Busy',
    'Restless', 'Easily_Distracted', 'Bothered_Worries', 'Hard_Concentrate',
    'Compare_Others', 'Feel_Comparisons', 'Seek_Validation', 'Feel_Depressed',
    'Interest_Fluctuate', 'Sleep_Issues'
]

In [ ]:
# checking for missing values
df.isnull().sum()

## Preprocessing

Organization had 30 missing values, everthing else was fine. Heres what I did to clean it up:
- Dropped Timestamp and Uses_SM (timestamp is just when they took the survey and everyone said yes to using social media so that column is useless)
- Dropped rows with missing values
- Converted Avg_Time from text to numbers with a dictionary. It had stuff like "Between 2 and 3 hours" so I mapped that to 2.5
- Turned Gender into numbers so the model can use it
- Counted how many platforms each person uses by splitting the comma list
- Dropped the remaining text columns since models cant take text

In [ ]:
df = df.drop(columns=['Timestamp', 'Uses_SM'])
df = df.dropna()

# avg time was text, need numbers
time_map = {
    'Less than an Hour': 0.5, 'Between 1 and 2 hours': 1.5,
    'Between 2 and 3 hours': 2.5, 'Between 3 and 4 hours': 3.5,
    'Between 4 and 5 hours': 4.5, 'More than 5 hours': 5.5
}
df['Avg_Time'] = df['Avg_Time'].map(time_map)

df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1, 'Nonbinary': 2, 'Transgender': 2})

# count how many platforms they use
df['Num_Platforms'] = df['Platforms'].str.split(',').str.len()

df = df.drop(columns=['Platforms', 'Organization', 'Relationship', 'Occupation'])
df = df.dropna()

### Target variable

The dataset didnt have a mental health label built in so I made one. I added up 5 of the mental health questions into a score and split at the median. The 5 questions were about worrying, concentration, depression, interest in activities, and sleep issues. Each one is 1-5 so the total goes from 5 to 25.

I used the median (17) as the cutoff because it gave me a pretty even split between the two groups.

In [ ]:
df['MH_Score'] = (df['Bothered_Worries'] + df['Hard_Concentrate'] +
                  df['Feel_Depressed'] + df['Interest_Fluctuate'] + df['Sleep_Issues'])

median_score = df['MH_Score'].median()
df['High_Impact'] = (df['MH_Score'] > median_score).astype(int)

print(f'Median: {median_score}')
print(df['High_Impact'].value_counts())

243 low impact and 203 high impact. Thats close enough to balanced.

I didnt use these 5 questions as features because they went into creating the target so that would be like giving the model the answer.

### Exploratory charts

Made a few charts before jumping into the models.

In [ ]:
sns.set_style('whitegrid')

# target distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='High_Impact', data=df)
plt.title('Distribution of Mental Health Impact')
plt.xlabel('0=Low, 1=High')
plt.tight_layout()
plt.show()

In [ ]:
# does time on social media matter?
plt.figure(figsize=(8, 5))
sns.boxplot(x='High_Impact', y='Avg_Time', data=df)
plt.title('Social Media Time vs Mental Health Impact')
plt.xlabel('0=Low, 1=High')
plt.ylabel('Hours per day')
plt.tight_layout()
plt.show()

The boxplot shows high impact people do spend a bit more time on social media but theres alot of overlap so time by itself probaly wont be enough to predict anything.

In [ ]:
# correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

Some of the features are pretty correlated with eachother. Like Compare_Others and Seek_Validation are at 0.43 which makes sense if you think about it.

## Modeling

I picked 11 features for the model. These are all the social media behavior columns plus demographics. I left out the mental health questions that I used to make the target variable.

Split the data 80/20. Used stratify so both sets have the same ratio of high vs low. Then scaled everything with StandardScaler because age and the survey questions are on totally diffrent scales.

In [ ]:
feature_cols = ['Age', 'Gender', 'Avg_Time', 'Num_Platforms', 'Purposeless',
                'Distracted_Busy', 'Restless', 'Easily_Distracted',
                'Compare_Others', 'Feel_Comparisons', 'Seek_Validation']

X = df[feature_cols]
y = df['High_Impact']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training: {X_train.shape[0]} rows')
print(f'Testing: {X_test.shape[0]} rows')

### Three models

I tried Logistic Regression, Decision Tree, and Random Forest. These are the ones we went over in class so I felt most comfortable with them. Used 5-fold cross validation to compare.

In [ ]:
# logistic regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr_scores = cross_val_score(lr, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f'Logistic Regression: {lr_scores.mean():.4f}')

# decision tree
dt = DecisionTreeClassifier(random_state=42, max_depth=5)
dt_scores = cross_val_score(dt, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f'Decision Tree: {dt_scores.mean():.4f}')

# random forest
rf = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=5)
rf_scores = cross_val_score(rf, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f'Random Forest: {rf_scores.mean():.4f}')

In [ ]:
# compare them
plt.figure(figsize=(8, 5))
plt.bar(['Logistic Regression', 'Decision Tree', 'Random Forest'],
        [lr_scores.mean(), dt_scores.mean(), rf_scores.mean()],
        color=['#2196F3', '#FF9800', '#4CAF50'])
plt.title('Model Comparison (CV Accuracy)')
plt.ylabel('Accuracy')
plt.ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

Logistic Regression won with 0.77. I honestly expected Random Forest to do better but I think the dataset is just too small for it to matter. With only 446 rows the simpler model did fine.

## Results

Trained LR on all the training data and tested on the test set.

In [ ]:
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)

print(f'Test Accuracy: {lr.score(X_test_scaled, y_test):.4f}')
print()
print(classification_report(y_test, lr_pred, target_names=['Low Impact', 'High Impact']))

76.67% on the test set which is pretty close to the 77% from cross validation so thats good. Means its not just memorizing the training data.

In [ ]:
# confusion matrix
cm = confusion_matrix(y_test, lr_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low', 'High'], yticklabels=['Low', 'High'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

It got 40 low impact people right and 29 high impact right. Missed 12 high impact people (predicted them as low) and 9 low impact people (predicted them as high). Its a little better at catching the low impact group then the high impact one.

In [ ]:
# feature importance (using random forest since LR doesnt have this)
rf.fit(X_train_scaled, y_train)
feat_imp = pd.DataFrame({'Feature': feature_cols, 'Importance': rf.feature_importances_})
feat_imp = feat_imp.sort_values('Importance', ascending=True)

plt.figure(figsize=(8, 6))
plt.barh(feat_imp['Feature'], feat_imp['Importance'])
plt.title('Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

Easily_Distracted was by far the biggest predictor, then Compare_Others after that. Stuff like Gender and number of platforms barely did anything. This was probaly the most intresting finding because it means its not really about how long you spend on social media, its about what your doing while your on there. Like if your constantly comparing yourself to people online thats way worse then just scrolling.

## Limitations

- Only 481 responses so the dataset is kinda small. More data would probaly help
- The target variable is something I made up by combining questions so someone else might do it diffrent and get diffrent results
- Its self reported so people might not be honest or accurate about how much they use social media
- 77% accuracy means its wrong about 1 in 4 times

**Ethical note:** this should not be used to diagnose anyone. Its just patterns in survey data, not a medical tool.

If I had more time I would try more models and maybe find a bigger dataset. I also think it would be cool to look at specific platforms instead of just counting how many someone uses.

## References

- Dataset: Souvik Ahmed, "Social Media and Mental Health" on Kaggle
- scikit-learn docs: https://scikit-learn.org/stable/user_guide.html
- pandas docs: https://pandas.pydata.org/docs/
- seaborn docs: https://seaborn.pydata.org/tutorial.html
- DTSC 2301 lectures and demos

## AI Transparency

I used Claude (claude.ai) to help me debug errors when my code wasnt running and to clean up wording on the written sections. I referenced the sklearn and pandas docs for the actual code. Claude also helped with the website HTML/CSS. The analysis decisions and interpretations are mine.